Paquetes necesarios

In [2]:
import cv2
import time
import numpy as np

**TAREA**: Los ejemplos ilustrativos anteriores permiten saber el número de monedas presentes en la imagen ideal proporcionada. ¿Cómo determinar la cantidad de dinero presente en ella? A partir únicamente de las dimensiones, una sugerencia implicaría, en primer término, identificar de forma interactiva (por ejemplo haciendo clic en la imagen) una moneda de un valor fijo en la imagen (por ejemplo de 1€) para permitir establecer una relación entre píxeles y milímetros. Tras obtener esa información y las dimensiones en milímetros de las distintas monedas, es factible calcular la cantidad de dinero en la imagen ideal.

Una vez resuelto el reto con la imagen ideal proporcionada, diseña un demostrador en vivo que a partir de la entrada de vídeo de la cámara, determine el dinero presente en la escena. Si bien no hay restricciones sobre utilizar medidas geométricas o de color, o cualquier clasificador, no deben hacer uso de modelos de aprendizaje profundo. ¿Funciona correctamente? ¿Se observan problemas?
Posibles extras serían la presencia de objetos que no son monedas y/o haya solape entre las monedas.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo

Nota: Para establecer la correspondencia entre píxeles y milímetros, comentar que la moneda de un euro tiene un diámetro de 23.25 mm. la de 50 céntimos de 24.35, la de 20 céntimos de 22.25, etc. 



In [35]:
pixel_size_buffer = []
pixel_size = 0 # mm/pixel

CALIBRATION_SS = 5
start_time = time.time()

amounts_read = []

font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 2
thickness = 1
top_margin = 20

MONEY = {
    'cobre':   {8.125: 0.01, 9.375: 0.02, 10.625: 0.05},
    'dorado':  {9.875: 0.10, 11.125: 0.20, 12.125: 0.50},
    'bimetal': {11.625: 1.0, 12.875: 2.0},
    'billete': {(62, 120): 5, (67, 127): 10, (72, 133): 20, (77, 140): 50, (82, 147): 100, (82, 153): 200, (82, 160): 500}
}
TOLERANCE_MM = 0.8
TOLERANCE_BILL_MM = (2, 3)
H_LIMITE = 15

BORDER_PX = 20        # grosor de la franja de borde usada como muestra de fondo
DIST_MAX = 40.0       # distancia a/b que se mapea a 255 (ajustar mirando dist_u8)
MIN_SIGNAL = 8.0      # si ni el percentil 99.5 llega aquí, se considera que no hay objetos
L_WEIGHT = 0.5      # peso de la luminosidad frente a la crominancia
L_DEADZONE = 10.0   # diferencia de L (escala 0-255) que se ignora
CLOSE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
OPEN_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))

def segment(frame):
    # 1. Desenfoque para reducir el ruido de croma
    blurred = cv2.GaussianBlur(frame, (5, 5), 0)

    # 2. Lab en uint8 -> float32 para poder restar con signo
    lab = cv2.cvtColor(blurred, cv2.COLOR_BGR2LAB).astype(np.float32)
    l = lab[:, :, 0]
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    # 3. Color del fondo: mediana de a y b en la franja de borde
    border = np.zeros(a.shape, dtype=bool)
    border[:BORDER_PX, :] = True
    border[-BORDER_PX:, :] = True
    border[:, :BORDER_PX] = True
    border[:, -BORDER_PX:] = True
    l_bg = np.median(l[border])
    a_bg = np.median(a[border])
    b_bg = np.median(b[border])

    # 4. Distancia de crominancia al fondo
    dist_chroma = cv2.magnitude(a - a_bg, b - b_bg)
    dist_l = np.maximum(np.abs(l - l_bg) - L_DEADZONE, 0) * L_WEIGHT
    dist = np.maximum(dist_chroma, dist_l)

    # 5. Escalado con rango fijo a uint8
    dist_u8 = np.clip(dist * (255.0 / DIST_MAX), 0, 255).astype(np.uint8)

    # 6. Si no hay señal suficiente, devolvemos máscara vacía (evita segmentar ruido)
    if np.percentile(dist, 99.5) < MIN_SIGNAL:
        return np.zeros(a.shape, dtype=np.uint8), dist_u8

    # 7. Otsu con objetos en blanco (sin INV)
    _, binary = cv2.threshold(dist_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # 8. Cierre para rellenar huecos (logo, parte plateada) y apertura para quitar motas
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, CLOSE_KERNEL)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, OPEN_KERNEL)

    return binary, dist_u8

def meassure_color(hsv, mask, dist_img, max_val):
    # Píxeles lejos del borde
    center = ((dist_img > 0.75 * max_val) & (mask > 0)).astype(np.uint8) * 255
    # Píxeles cerca del borde, pero sin tocarlo
    ring = ((dist_img > 0.08 * max_val) & (dist_img < 0.25 * max_val)
              & (mask > 0)).astype(np.uint8) * 255

    h_c, s_c, _, _ = cv2.mean(hsv, mask=center)
    h_r, s_r, _, _ = cv2.mean(hsv, mask=ring)
    return (h_c, s_c), (h_r, s_r)

def classify_group(centro, anillo, umbral_bimetal=12):
    (h_c, s_c), (h_r, s_r) = centro, anillo

    # Monedas con dos tonos (1€ y 2€) tienen un umbral de diferencia de tono más alto
    if abs(h_c - h_r) > umbral_bimetal or abs(s_c - s_r) > 40:
        return 'bimetal'

    # Monedas de un solo tono se clasifican en cobre o dorado según el valor medio del tono
    h = (h_c + h_r) / 2
    return 'cobre' if h < H_LIMITE else 'dorado'

def classify(radius, group):
    # Buscamos la moneda más cercana al radio medido para el grupo determinado
    key = min(MONEY[group], key=lambda k: abs(k - radius))
    if abs(key - radius) > TOLERANCE_MM:
        return None          # no encaja con ninguna moneda del grupo
    return MONEY[group][key]

def create_countour_elements(binary_image, index):
    # Creamos una máscara para el contorno actual y calculamos el valor máximo de la transformada de distancia dentro de esa máscara
    mask = np.zeros(binary_image.shape, dtype=np.uint8)
    cv2.drawContours(mask, contours, index, 255, thickness=-1)
    distance_image = cv2.distanceTransform(mask, cv2.DIST_L2, 5)

    return mask, distance_image

def find_distance_from_center_px(contour, binary_image, index):
    # Eliminamos ruido de contornos pequeños
    if cv2.contourArea(contour) < 100:
        return None

    mask, distance_image = create_countour_elements(binary_image, index)

    _, distance_from_center, _, _ = cv2.minMaxLoc(distance_image, mask=mask)

    return distance_from_center

def calibrate(contours, binary_image):
    index, max_contour = max(enumerate(contours), key=lambda par: cv2.contourArea(par[1]))
    distance = find_distance_from_center_px(max_contour, binary_image, index)

    if distance:
        # Calculamos el tamaño de pixel a partir del radio de referencia y lo añadimos al buffer
        pixel_size_buffer.append(REFERENCE_OBJECT_MM / distance)

def classify_bill(short, long, group):
    # Buscamos el billete cuyo tamaño sea el más cercano al del contorno detectado
    key = min(MONEY[group], key=lambda k: abs(k[0] - short) + abs(k[1] - long))
    if abs(key[0] - short) > TOLERANCE_BILL_MM[0] and abs(key[1] - long) > TOLERANCE_BILL_MM[1]:
        return None          # no encaja con ningún billete
    return MONEY[group][key]

def get_bill_shape(contour):
    # Obtenemos el ancho y largo del contorno del billete
    _, (w, h), _ = cv2.minAreaRect(contour)

    # Ordenamos w y h para obtener siempre el mismo orden, sin importar el ángulo
    short_px, long_px = sorted((w, h))

    # Conversión a mm con la escala de la calibración
    short_mm = short_px * pixel_size
    long_mm = long_px * pixel_size

    # Relación de aspecto (con protección frente a división por cero)
    aspect = long_px / short_px if short_px > 0 else 0

    return short_mm, long_mm, aspect

def get_amount(contours, binary_image, hsv, frame, title_position, debugging):
    total_amount = 0
    for index, contour in enumerate(contours):
        distance = find_distance_from_center_px(contour, binary_image, index)

        if distance is None or distance <= 0:
            continue

        mask, distance_image = create_countour_elements(binary_image, index)

        # Calculamos el radio en mm a partir del valor máximo de la transformada de distancia y el tamaño de pixel
        radius = distance * pixel_size

        text = ""
        if radius > 20:
            short_mm, long_mm, aspect = get_bill_shape(contour)

            # Utilizamos la relación de aspectos para evitar leer elementos que no sean similares a billetes
            if aspect < 1.7 or aspect > 2.05:
                continue

            value = classify_bill(short_mm, long_mm, 'billete')

            if value is not None:
                total_amount += value
                if debugging:
                    text = f'Billete: {value}€ - Tamaño: {short_mm:.2f}x{long_mm:.2f}mm'
                else:
                    text = f"Billete: {value}€"
            else:
                if debugging:
                    text = f'Billete desconocido - Tamaño: {short_mm:.2f}x{long_mm:.2f}mm'
                else:
                    text = "Billete desconocido"
            draw_text(frame, text, font, 0.5, (0,0,0), thickness, top_margin, get_text_position(contour))

        elif radius > 7:
            # Establecemos el grupo al que pertenece la moneda en base a su color
            measurements = meassure_color(hsv, mask, distance_image, distance)
            group = classify_group(*measurements)

            if group is not None:
                # Clasificamos la moneda y sumamos su valor al total de la mesa
                value = classify(radius, group)
                if value is not None:
                    total_amount += value
                    if debugging:
                        text = f'Moneda: {value}€ - Radio: {radius:.2f}mm - Grupo: {group}'
                    else:
                        text = f'Moneda: {value}€'
                else:
                    if debugging:
                        text = f'Moneda desconocida - Radio: {radius:.2f}mm - Grupo: {group}'
                    else:
                        text = "Moneda desconocida"
                draw_text(frame, text, font, 0.5, (0,0,0), thickness, top_margin, get_text_position(contour))
                

    # Añadimos el total de la mesa a la lista de lecturas
    amounts_read.append(total_amount)
    # Se muestra el valor de la mesa como la mediana de las lecturas obtenidas
    text = f'Valor en la mesa: {np.median(amounts_read)}'
    draw_text(frame, text, font, font_scale, (0,0,0), thickness, top_margin, title_position)

def draw_text(frame, text, font, font_scale, color, thickness, top_margin=0, position=None):
    _, w = frame.shape[:2]

    (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)

    if position is None:
        position = (None, None)
    px, py = position

    # Eje X
    if px is None:
        x = (w - text_w) // 2          # centrado en el frame
    else:
        x = px - text_w // 2           # centrado en la posición dada

    # Eje Y
    if py is None:
        y = top_margin + text_h
    else:
        y = py + text_h // 2           # centro vertical del texto en py

    cv2.putText(frame, text, (x, y), font, font_scale, color, thickness)

def get_text_position(contour, offset=10):
    x, y, w, _ = cv2.boundingRect(contour)
    cx = x + w // 2        # centro horizontal del objeto
    cy = y - offset        # justo encima del objeto
    return (cx, cy)

### Aplicación a imagen ideal

In [33]:
REFERENCE_OBJECT_MM = 12.875

pixel_size_buffer = []
pixel_size = 0 # mm/pixel

amounts_read = []

font_scale = 1

# Leemos la entrada de la cámara fotograma a fotograma
image = cv2.imread("./images/Monedas.jpg")

if image is not None:
    # Binarizamos la imagen y calculamos los contornos en esta
    binarized_image, _ = segment(image)
    contours, _ = cv2.findContours(binarized_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Dibujamos los contornos sobre la imagen original para mostrar al usuario
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    cv2.drawContours(rgb_image, contours, -1, (0,255,0), 3)
    bgr_frame = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)

    # Leemos el radio del contorno de referencia y establecemos un tamaño de pixel
    calibrate(contours, binarized_image)

    # Calculamos el tamaño de pixel a partir de la mediana de las lecturas previas
    pixel_size = pixel_size_buffer[0]

    # Nos aseguramos de que el tamaño de pixel no sea cero antes de continuar con la medición
    if pixel_size == 0:
        print("Error: El tamaño de pixel es cero. La calibración puede haber fallado.")
    else:
      h, _ = bgr_frame.shape[:2]
      position = (None, h-30)
      get_amount(contours, binarized_image, hsv, bgr_frame, position, False)
      cv2.imwrite("./images/Monedas_resultado.jpg", bgr_frame)

### Aplicación a la captura en vídeo

In [37]:
REFERENCE_OBJECT_MM = 27

pixel_size_buffer = []
pixel_size = 0 # mm/pixel

CALIBRATION_SS = 5
start_time = time.time()

amounts_read = []

font_scale = 2
H_LIMITE = 23

vid = cv2.VideoCapture(1)

while(True):
    # Leemos la entrada de la cámara fotograma a fotograma
    ret, image = vid.read()

    if ret:
        # Binarizamos la imagen y calculamos los contornos en esta
        binarized_image, _ = segment(image)
        contours, _ = cv2.findContours(binarized_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Dibujamos los contornos sobre la imagen original para mostrar al usuario
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        cv2.drawContours(rgb_image, contours, -1, (0,255,0), 3)
        bgr_frame = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)

        # Calibramos el fondo durante los primeros segundos
        if time.time() - start_time < CALIBRATION_SS:
            # Leemos el radio del contorno de referencia y establecemos un tamaño de pixel
            calibrate(contours, binarized_image)

            # Calculamos el tamaño de pixel a partir de la mediana de las lecturas previas
            pixel_size = np.median(pixel_size_buffer)

            # Mostramos los fotogramas para aportar feedback visual al usuario
            cv2.imshow('Fotograma', bgr_frame)

            # Detenemos pulsando ESC
            if cv2.waitKey(20) == 27:
                break

            continue
        # Nos aseguramos de que el tamaño de pixel no sea cero antes de continuar con la medición
        if pixel_size == 0:
            print("Error: El tamaño de pixel es cero. La calibración puede haber fallado.")
            break

        position = None
        get_amount(contours, binarized_image, hsv, bgr_frame, position, False)
        # Mostramos los fotogramas para aportar feedback visual al usuario
        cv2.imshow('Fotograma', bgr_frame)

    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

# Liberamos el objeto de captura y destruimos las ventanas
vid.release()
cv2.destroyAllWindows()

**TAREA**: La tarea consiste en extraer características (geométricas y/o visuales) de las tres imágenes completas de partida, y *aprender* patrones que permitan identificar las partículas en nuevas imágenes. Para ello se proporciona como imagen de test *MPs_test.jpg* y sus correpondientes anotaciones *MPs_test_bbs.csv* con la que deben obtener las métricas para su propuesta de clasificación de microplásticos, además de la matriz de confusión. La matriz de confusión permitirá mostrar para cada clase el número de muestras que se clasifican correctamente de dicha clase, y el número de muestras que se clasifican incorrectamente como perteneciente a una de las otras dos clases. De nuevo, no deben hacer uso de modelos de aprendizaje profundo ni para segmentar, ni para clasificar.

En el trabajo [SMACC: A System for Microplastics Automatic Counting and Classification](https://doi.org/10.1109/ACCESS.2020.2970498), las características geométricas utilizadas fueron:

- Área en píxeles
- Perímetro en píxeles
- Compacidad (relación entre el cuadrado del perímetro y el área de la partícula)
- Relación del área de la partícula con la del contenedor
- Relación del ancho y el alto del contenedor
- Relación entre los ejes de la elipse ajustada
- Definido el centroide, relación entre las distancias menor y mayor al contorno

Si no se quedan satisfechos con la segmentación obtenida, es el mundo real, también en el README comento técnicas recientes de segmentación, que podrían despertar su curiosidad.

IMPORTANTE: No se admite el uso de modelos de aprendizaje profundo

Sección 1: Importaciones, Configuración e Ingesta de Datos

In [ ]:
import os
import csv
import warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.filters import threshold_sauvola
from skimage.feature import local_binary_pattern
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,          # Exactitud global
    balanced_accuracy_score, # Exactitud balanceada
    precision_score,         # Precisión
    recall_score,            # Sensibilidad
    f1_score,                # Medida F1
    confusion_matrix,        # Matriz de confusión
    classification_report    # Informe detallado de clasificación
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Ignoramos advertencias para mantener limpia la salida por consola
warnings.filterwarnings("ignore")

# Imágenes completas utilizadas como referencia para cada
# tipo de microplástico presente en el conjunto de entrenamiento.
#
# FRA = Fragment
# PEL = Pellet
# TAR = Tar
FULL_TRAIN_PATHS = {
    "FRA": "fragment-03-olympus-10-01-2020.jpg",
    "PEL": "pellet-03-olympus-10-01-2020.jpg",
    "TAR": "tar-03-olympus-10-01-2020.jpg",
}

# Subimágenes previamente recortadas que contienen
# exclusivamente el objeto de interés de cada clase.
SUB_TRAIN_PATHS = {
    "FRA": "FRA.png",
    "PEL": "PEL.png",
    "TAR": "TAR.png",
}

# Imagen de prueba donde se realizará la detección
# y clasificación automática de microplásticos.
TEST_IMG_FILE = "MPs_test.jpg"

# Archivo CSV que contiene las cajas delimitadoras
# anotadas manualmente para validar los resultados.
TEST_CSV_FILE = "MPs_test_bbs.csv"

# Semilla para garantizar reproducibilidad en procesos aleatorios.
SEED_VALUE = 13

# Las imágenes originales poseen una resolución elevada.
# Se reducen para acelerar el procesamiento manteniendo
# suficiente detalle para detectar los microplásticos.
TRAIN_RESIZE_SCALE = 0.25

# Tamaño de la ventana local utilizada por Sauvola
# para calcular el umbral adaptativo.
SAUVOLA_WINDOW_SIZE = 51

# Parámetro de sensibilidad del método de Sauvola.
SAUVOLA_K_PARAM = 0.06

# Área mínima y máxima aceptada para regiones detectadas
# en imágenes completas.
AREA_LIMITS_FULL = (42, 3500)

# Área mínima y máxima aceptada para regiones detectadas
# en subimágenes.
AREA_LIMITS_SUB = (24, 3500)

# Número de imágenes sintéticas generadas mediante
# aumento de datos para cada muestra de entrenamiento.
AUGMENTATION_COUNT = 5

# Solapamiento mínimo (IoU) exigido para considerar
# una detección como correcta.
MIN_IOU_THRESHOLD = 0.30

# Inicialización de la semilla aleatoria.
np.random.seed(SEED_VALUE)

# Carga una imagen desde la carpeta images y comprueba
# que el archivo existe y puede abrirse correctamente.
def read_image_file(file_path):

    # Construye la ruta completa de la imagen.
    img_path = os.path.join("images", file_path)

    # Comprueba que el archivo existe.
    if not os.path.exists(img_path):
        raise FileNotFoundError(
            f"No se encontró la imagen: {img_path}"
        )

    # Lee la imagen utilizando OpenCV.
    img = cv2.imread(img_path)

    # Verifica que la lectura se haya realizado correctamente.
    if img is None:
        raise ValueError(
            f"No se pudo cargar la imagen: {img_path}"
        )

    return img

# Carga una imagen de entrenamiento y reduce su tamaño
# cuando sea necesario para disminuir el coste computacional.
def read_and_scale_training_image(file_path, is_sub_image=False):

    # Carga la imagen desde disco.
    raw_img = read_image_file(file_path)

    # Si la imagen completa ya tiene un tamaño moderado,
    # se utiliza sin aplicar escalado.
    if not is_sub_image and max(raw_img.shape[:2]) <= 1200:
        return raw_img

    # Redimensionamiento proporcional conservando
    # la relación de aspecto original.
    return cv2.resize(
        raw_img,
        None,
        fx=TRAIN_RESIZE_SCALE,
        fy=TRAIN_RESIZE_SCALE,
        interpolation=cv2.INTER_AREA
    )

print("=== SECCIÓN 1: COMPROBACIÓN DE ARCHIVOS Y DIMENSIONES ===")

# Almacena información descriptiva de cada imagen cargada.
info_rows = []

# Procesa cada una de las clases de microplásticos.
for cls_key in ["FRA", "PEL", "TAR"]:

    # Carga la imagen completa correspondiente.
    img_f = read_and_scale_training_image(
        FULL_TRAIN_PATHS[cls_key],
        is_sub_image=False
    )

    # Carga la subimagen correspondiente.
    img_s = read_and_scale_training_image(
        SUB_TRAIN_PATHS[cls_key],
        is_sub_image=True
    )

    # Guarda las dimensiones para mostrarlas posteriormente.
    info_rows.append({
        "Clase": cls_key,
        "Imagen Completa (px)": (
            f"{img_f.shape[1]}x{img_f.shape[0]}"
        ),
        "Subimagen (px)": (
            f"{img_s.shape[1]}x{img_s.shape[0]}"
        ),
    })

# Muestra un resumen tabulado para verificar que todas
# las imágenes se han cargado correctamente.
print(pd.DataFrame(info_rows).to_string(index=False))

Sección 2: Segmentación Híbrida (CIE Lab + Pseudo-Transparencia + Sauvola + Otsu Local)

In [ ]:
# Calcula la distancia euclídea entre dos imágenes considerando
# simultáneamente los tres canales de color.
def compute_channel_distance(img_array, bg_array):

    # Conversión a coma flotante para evitar pérdidas de precisión
    # durante el cálculo de diferencias entre píxeles.
    diff_sq = (img_array.astype(np.float32) - bg_array.astype(np.float32)) ** 2

    # Distancia euclídea entre los vectores de color.
    return np.sqrt(np.sum(diff_sq, axis=-1))


# Estima un fondo limpio en espacio CIE Lab para resaltar
# las partículas y reducir artefactos del escáner.
def estimate_clean_lab_background(bgr_img):

    # Suaviza el ruido preservando los bordes de las partículas.
    filtered_bgr = cv2.bilateralFilter(bgr_img, 7, 40, 7)

    # Conversión a CIE Lab para separar mejor las diferencias
    # de luminosidad y color respecto al fondo.
    lab_img = cv2.cvtColor(filtered_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    # Estimación inicial del fondo mediante filtrado mediano.
    # El tamaño grande de la ventana elimina las partículas
    # conservando únicamente las variaciones globales.
    median_planes = [
        cv2.medianBlur(ch, 61)
        for ch in cv2.split(lab_img.astype(np.uint8))
    ]
    median_bg = cv2.merge(median_planes).astype(np.float32)

    # Detección preliminar de regiones distintas del fondo.
    prelim_dist = compute_channel_distance(lab_img, median_bg)
    prelim_mask = (prelim_dist > 8.0).astype(np.uint8)

    # Expansión de la máscara para cubrir completamente
    # las partículas y posibles halos circundantes.
    dilated_mask = cv2.dilate(
        prelim_mask,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)),
        iterations=1
    )

    # Sustituye las zonas ocupadas por partículas por la
    # estimación del fondo obtenida anteriormente.
    inpainted_bg = np.where(
        dilated_mask[:, :, None] > 0,
        median_bg,
        lab_img
    )

    # Suavizado adicional para eliminar bandas horizontales
    # y variaciones lentas introducidas por el escáner.
    smooth_bg = cv2.GaussianBlur(inpainted_bg, (31, 15), 0)

    # Mide la diferencia cromática entre la imagen original
    # y el fondo estimado.
    final_lab_dist = compute_channel_distance(lab_img, smooth_bg)

    # Genera un mapa donde los objetos destacan frente al fondo,
    # facilitando la segmentación posterior.
    pseudo_trans_map = np.clip(
        255.0 - final_lab_dist * 2.0,
        0,
        255
    ).astype(np.uint8)

    return filtered_bgr, lab_img, smooth_bg, pseudo_trans_map


# Refina una máscara obtenida con Sauvola utilizando
# información de contraste local basada en Otsu.
def refine_mask_with_local_otsu(pseudo_crop, sauvola_crop_mask):

    # Inversión del contraste para destacar las partículas.
    contrast_crop = 255 - pseudo_crop

    # Si la región no contiene píxeles segmentados,
    # no es necesario realizar refinamiento.
    if not np.any(sauvola_crop_mask > 0):
        return sauvola_crop_mask

    # Contraste máximo presente en la partícula candidata.
    max_contrast = float(
        np.max(contrast_crop[sauvola_crop_mask > 0])
    )

    # Evita refinar regiones con escaso contraste.
    if max_contrast <= 50.0:
        return sauvola_crop_mask

    # Obtención automática del umbral mediante Otsu.
    otsu_thresh, _ = cv2.threshold(
        contrast_crop,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Ajuste del umbral para adaptarlo al contraste real
    # de la región analizada.
    effective_thresh = max(
        float(otsu_thresh) * 0.70,
        max_contrast * 0.30
    )

    # Nueva segmentación basada en el contraste local.
    _, otsu_mask = cv2.threshold(
        contrast_crop,
        effective_thresh,
        255,
        cv2.THRESH_BINARY
    )

    # Conserva únicamente los píxeles aceptados tanto por
    # Sauvola como por el refinamiento de Otsu.
    combined = cv2.bitwise_and(
        sauvola_crop_mask,
        otsu_mask
    )

    # Evita generar regiones demasiado pequeñas.
    if np.sum(combined > 0) < 15:
        return sauvola_crop_mask

    # Devuelve la máscara refinada.
    return combined


# Segmenta las partículas combinando estimación de fondo,
# Sauvola y refinamiento local mediante Otsu.
def segment_particles(
    bgr_img,
    is_sub_image=False,
    k_val=SAUVOLA_K_PARAM
):

    # Obtención de la imagen filtrada y del mapa de
    # pseudo-transparencia utilizado para segmentar.
    filtered_bgr, lab_img, smooth_bg, pseudo_trans = (
        estimate_clean_lab_background(bgr_img)
    )

    # Umbralización adaptativa de Sauvola.
    sauvola_thresh_map = threshold_sauvola(
        pseudo_trans,
        window_size=SAUVOLA_WINDOW_SIZE,
        k=k_val
    )

    # Generación de una máscara binaria inicial donde las
    # partículas quedan separadas del fondo.
    init_mask = (
        pseudo_trans < sauvola_thresh_map
    ).astype(np.uint8) * 255

    # Eliminación de ruido y relleno de pequeños huecos.
    morph_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (3, 3)
    )

    # Apertura morfológica para eliminar pequeñas regiones
    # aisladas y ruido generado durante la umbralización.
    init_mask = cv2.morphologyEx(
        init_mask,
        cv2.MORPH_OPEN,
        morph_kernel
    )

    # Cierre morfológico para rellenar pequeños huecos y unir
    # fragmentos próximos pertenecientes a una misma partícula.
    init_mask = cv2.morphologyEx(
        init_mask,
        cv2.MORPH_CLOSE,
        morph_kernel
    )

    # Extracción de contornos candidatos.
    raw_contours, _ = cv2.findContours(
        init_mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    # Obtención de las dimensiones de la imagen para
    # validar posteriormente la posición de los contornos.
    img_h, img_w = bgr_img.shape[:2]

    # Selección de límites de área según el tipo de imagen.
    min_area, max_area = (
        AREA_LIMITS_SUB
        if is_sub_image
        else AREA_LIMITS_FULL
    )

    # Lista donde se almacenarán los contornos válidos
    # tras todas las etapas de filtrado y refinamiento.
    detected_contours = []

    # Procesa individualmente cada región detectada.
    for cnt in raw_contours:

        # Filtrado de contornos demasiado pequeños,
        # demasiado grandes o mal definidos.
        cnt_area = cv2.contourArea(cnt)

        if not (min_area <= cnt_area <= max_area) or len(cnt) < 5:
            continue

        # Obtiene el rectángulo mínimo que contiene
        # completamente el contorno actual.
        bx, by, bw, bh = cv2.boundingRect(cnt)

        if not is_sub_image:

            # Eliminación de artefactos generados por el
            # borde del escáner o de la imagen.
            if by < 15 or (
                by + bh >= img_h - 2
                and (bh < 8 or bw > 60)
            ):
                continue

        # Ampliación local para refinar la región.
        pad = 2

        # Coordenadas de la ventana local utilizada
        # durante el refinamiento del contorno.
        x0, y0 = max(0, bx - pad), max(0, by - pad)
        x1, y1 = min(img_w, bx + bw + pad), min(img_h, by + bh + pad)

        # Máscara local asociada al contorno actual.
        local_sauvola = np.zeros(
            (y1 - y0, x1 - x0),
            dtype=np.uint8
        )

        # Dibuja el contorno actual sobre una máscara local
        # rellena. La traslación de coordenadas permite que
        # el contorno encaje correctamente dentro de la región
        # recortada utilizada para el refinamiento.
        cv2.drawContours(
            local_sauvola,
            [cnt - np.array([[x0, y0]])],
            -1,
            255,
            -1
        )

        # Extrae la región correspondiente del mapa
        # de pseudo-transparencia.
        local_pseudo = pseudo_trans[y0:y1, x0:x1]

        # Refinamiento local con Otsu.
        refined_local = refine_mask_with_local_otsu(
            local_pseudo,
            local_sauvola
        )

        # Aplica un cierre morfológico para unir pequeñas
        # discontinuidades producidas durante el refinamiento.
        refined_local = cv2.morphologyEx(
            refined_local,
            cv2.MORPH_CLOSE,
            morph_kernel
        )

        # Obtención de contornos refinados.
        sub_cnts, _ = cv2.findContours(
            refined_local,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        if sub_cnts:

            # Conserva el contorno dominante dentro
            # de la región analizada.
            best_cnt = max(
                sub_cnts,
                key=cv2.contourArea
            )

            # Conserva únicamente contornos que siguen
            # cumpliendo las restricciones geométricas.
            if (
                cv2.contourArea(best_cnt) >= (min_area - 5)
                and len(best_cnt) > 5
            ):
                # Reposiciona el contorno refinado respecto
                # al sistema de coordenadas de la imagen completa.
                detected_contours.append(
                    best_cnt + np.array([[x0, y0]])
                )

    # Construcción de la máscara binaria final.
    clean_binary_mask = np.zeros_like(init_mask)

    # Dibuja todas las partículas detectadas sobre una
    # máscara binaria final.
    cv2.drawContours(
        clean_binary_mask,
        detected_contours,
        -1,
        255,
        -1
    )

    # Almacena información adicional de la segmentación
    # para análisis y visualización posteriores.
    segmentation_meta = {
        "pseudo_trans": pseudo_trans,
        "mask_clean": clean_binary_mask,
        "sauvola_mean": float(np.mean(sauvola_thresh_map)),
    }

    return (
        detected_contours,
        lab_img,
        smooth_bg,
        filtered_bgr,
        segmentation_meta
    )

print("\n=== SECCIÓN 2: CONTEO DE PARTÍCULAS SEGMENTADAS EN ENTRENAMIENTO ===")

# Almacena estadísticas de segmentación para cada clase.
seg_summary = []

# Figura con dos filas y tres columnas:
# - Fila superior: imágenes completas.
# - Fila inferior: subimágenes.
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Procesa las tres clases de microplásticos.
for idx, cls_key in enumerate(["FRA", "PEL", "TAR"]):

    # Carga la imagen completa de entrenamiento.
    img_full = read_and_scale_training_image(
        FULL_TRAIN_PATHS[cls_key],
        is_sub_image=False
    )

    # Segmenta las partículas presentes en la imagen.
    cnts_full, _, _, _, meta_full = segment_particles(
        img_full,
        is_sub_image=False
    )

    # Copia de la imagen original para visualizar resultados.
    vis_full = img_full.copy()

    # Dibuja los contornos detectados en color rojo.
    cv2.drawContours(
        vis_full,
        cnts_full,
        -1,
        (0, 0, 255),
        2
    )

    # Muestra la imagen segmentada.
    axes[0, idx].imshow(
        cv2.cvtColor(vis_full, cv2.COLOR_BGR2RGB)
    )

    # Indica el número de partículas detectadas y el valor
    # medio del umbral calculado por Sauvola.
    axes[0, idx].set_title(
        f"Completa {cls_key}: {len(cnts_full)} partículas\n"
        f"(Sauvola m={meta_full['sauvola_mean']:.1f})"
    )

    axes[0, idx].axis("off")

    # Carga la subimagen asociada a la clase.
    img_sub = read_and_scale_training_image(
        SUB_TRAIN_PATHS[cls_key],
        is_sub_image=True
    )

    # Segmenta las partículas presentes en el recorte.
    cnts_sub, _, _, _, meta_sub = segment_particles(
        img_sub,
        is_sub_image=True
    )

    # Copia para visualización.
    vis_sub = img_sub.copy()

    # Dibuja los contornos detectados en color verde.
    cv2.drawContours(
        vis_sub,
        cnts_sub,
        -1,
        (0, 200, 0),
        2
    )

    # Muestra la subimagen segmentada.
    axes[1, idx].imshow(
        cv2.cvtColor(vis_sub, cv2.COLOR_BGR2RGB)
    )

    # Información resumida para la subimagen.
    axes[1, idx].set_title(
        f"Subimagen {cls_key}: {len(cnts_sub)} partículas\n"
        f"(Sauvola m={meta_sub['sauvola_mean']:.1f})"
    )

    axes[1, idx].axis("off")

    # Guarda estadísticas para la tabla resumen.
    seg_summary.append({
        "Clase": cls_key,
        "Partículas (Completa)": len(cnts_full),
        "Partículas (Subimagen)": len(cnts_sub),

        # Número total de partículas disponibles para la clase.
        "Total Base": len(cnts_full) + len(cnts_sub),

        # Valor medio del umbral adaptativo de Sauvola.
        "Media Sauvola": round(meta_full["sauvola_mean"], 2)
    })

# Muestra un resumen numérico de la segmentación obtenida.
print(pd.DataFrame(seg_summary).to_string(index=False))

# Título general de la figura.
plt.suptitle(
    "Segmentación Híbrida (CIE Lab + Pseudo-Transparencia + Sauvola + Otsu Local)",
    fontsize=14,
    y=0.98
)

# Ajusta manualmente la distribución de la figura para mejorar
# la separación entre el título y las imágenes, así como entre
# las distintas filas y columnas de subgráficas.
plt.subplots_adjust(
    top=0.88,
    hspace=0.40,
    wspace=0.20
)

# Muestra los resultados por pantalla.
plt.show()

Sección 3: Extracción de Características (Geometría SMACC + Apariencia CIE Lab + LBP)

In [ ]:
# Extrae características geométricas que describen la forma
# y dimensiones de una partícula segmentada.
def extract_geometric_features(contour):

    # Rectángulo delimitador mínimo alineado con los ejes.
    bx, by, bw, bh = cv2.boundingRect(contour)

    # Área y perímetro del contorno.
    area_val = float(cv2.contourArea(contour))
    perim_val = float(cv2.arcLength(contour, True))

    # Descriptores geométricos clásicos utilizados para
    # diferenciar partículas según su forma.
    compactness_val = (
        (perim_val ** 2) / area_val
        if area_val > 0 else 0.0
    )

    circularity_val = (
        (4.0 * np.pi * area_val) / (perim_val ** 2)
        if perim_val > 0 else 0.0
    )

    extent_val = (
        area_val / float(bw * bh)
        if (bw * bh) > 0 else 0.0
    )

    aspect_val = (
        float(min(bw, bh)) / float(max(bw, bh))
        if max(bw, bh) > 0 else 1.0
    )

    # Relación entre los ejes de la elipse ajustada al contorno.
    ellipse_ratio_val = 1.0

    # El ajuste elíptico requiere al menos cinco puntos.
    if len(contour) >= 5:
        try:

            # Ajuste de una elipse al contorno.
            (_, axes_len, _) = cv2.fitEllipse(contour)

            # Cálculo de la relación entre eje mayor y eje menor.
            if min(axes_len) > 0:
                ellipse_ratio_val = (
                    float(max(axes_len)) /
                    float(min(axes_len))
                )

        except cv2.error:
            pass

    # Obtención del centroide mediante momentos geométricos.
    moments = cv2.moments(contour)

    if moments["m00"] != 0:
        cx = moments["m10"] / moments["m00"]
        cy = moments["m01"] / moments["m00"]
    else:
        # Fallback usando el centro del rectángulo delimitador.
        cx = bx + bw / 2.0
        cy = by + bh / 2.0

    # Conversión de los puntos del contorno a coordenadas planas.
    pts = contour.reshape(-1, 2).astype(np.float32)

    # Distancia de cada punto al centroide.
    radial_dists = np.sqrt(
        (pts[:, 0] - cx) ** 2 +
        (pts[:, 1] - cy) ** 2
    )

    # Mide cuánto varía el radio de la partícula.
    radius_ratio_val = float(
        radial_dists.max() /
        max(radial_dists.min(), 1.0)
    )

    # Devuelve todos los descriptores geométricos calculados.
    return {
        "x": bx,
        "y": by,
        "w": bw,
        "h": bh,
        "area": area_val,
        "perimeter": perim_val,
        "circularity": circularity_val,
        "compactness": compactness_val,
        "extent": extent_val,
        "aspect_ratio": aspect_val,
        "ellipse_ratio": ellipse_ratio_val,
        "radius_ratio": radius_ratio_val,
    }

# Extrae descriptores de color y textura a partir de una
# partícula previamente segmentada.
def extract_appearance_and_texture_features(
    contour,
    filtered_bgr,
    lab_img,
    bg_lab_img
):

    # Obtención del rectángulo delimitador.
    bx, by, bw, bh = cv2.boundingRect(contour)

    # Añade un pequeño margen alrededor de la partícula.
    pad = 2

    x0, y0 = max(0, bx - pad), max(0, by - pad)
    x1, y1 = min(filtered_bgr.shape[1], bx + bw + pad), min(filtered_bgr.shape[0], by + bh + pad)

    # Construcción de la máscara binaria local.
    sub_mask = np.zeros((y1 - y0, x1 - x0), np.uint8)

    cv2.drawContours(
        sub_mask,
        [contour - np.array([[x0, y0]])],
        -1,
        255,
        -1
    )

    # Dilatación ligera para compensar pérdidas producidas
    # durante el refinamiento local con Otsu.
    color_mask = cv2.dilate(
        sub_mask,
        np.ones((3, 3), np.uint8),
        iterations=1
    ) > 0

    # Máscara original de primer plano.
    fg_mask = sub_mask > 0

    # Extracción de píxeles pertenecientes a la partícula
    # en la imagen Lab y en el fondo estimado.
    fg_lab = lab_img[y0:y1, x0:x1][color_mask].astype(np.float32)
    bg_lab = bg_lab_img[y0:y1, x0:x1][color_mask].astype(np.float32)

    # Relación de luminosidad respecto al fondo.
    l_ratio_val = float(
        fg_lab[:, 0].mean() /
        bg_lab[:, 0].mean()
    )

    # Diferencia cromática relativa frente al fondo.
    chroma_rel_val = float(
        compute_channel_distance(
            fg_lab[:, 1:],
            bg_lab[:, 1:]
        ).mean()
    )

    # Conversión a espacios RGB y HSV para obtener
    # estadísticas descriptivas de color.
    rgb_patch = cv2.cvtColor(
        filtered_bgr[y0:y1, x0:x1],
        cv2.COLOR_BGR2RGB
    )[fg_mask].astype(np.float32)

    hsv_patch = cv2.cvtColor(
        filtered_bgr[y0:y1, x0:x1],
        cv2.COLOR_BGR2HSV
    )[fg_mask].astype(np.float32)

    # Medias de los canales RGB.
    r_mean, g_mean, b_mean = np.mean(
        rgb_patch,
        axis=0
    ).tolist()

    # Varianzas de los canales RGB.
    r_var, g_var, b_var = np.var(
        rgb_patch,
        axis=0
    ).tolist()

    # Medias de los canales HSV.
    h_mean, s_mean, v_mean = np.mean(
        hsv_patch,
        axis=0
    ).tolist()

    # Varianzas de los canales HSV.
    h_var, s_var, v_var = np.var(
        hsv_patch,
        axis=0
    ).tolist()

    # Conversión a escala de grises para analizar textura.
    gray_crop = cv2.cvtColor(
        filtered_bgr[y0:y1, x0:x1],
        cv2.COLOR_BGR2GRAY
    )

    # Cálculo del patrón binario local (LBP).
    lbp_map = local_binary_pattern(
        gray_crop,
        P=8,
        R=1,
        method="default"
    )

    # Histograma normalizado de texturas.
    lbp_hist, _ = np.histogram(
        lbp_map[fg_mask],
        bins=256,
        range=(0, 256),
        density=True
    )

    # Diccionario con los descriptores de color.
    feat_dict = {
        "L_ratio": l_ratio_val,
        "chroma_rel": chroma_rel_val,
        "R_mean": r_mean,
        "G_mean": g_mean,
        "B_mean": b_mean,
        "R_var": r_var,
        "G_var": g_var,
        "B_var": b_var,
        "H_mean": h_mean,
        "S_mean": s_mean,
        "V_mean": v_mean,
        "H_var": h_var,
        "S_var": s_var,
        "V_var": v_var,
    }

    # Añade cada bin del histograma LBP como una característica.
    for idx_bin, bin_val in enumerate(lbp_hist):
        feat_dict[f"LBP_{idx_bin}"] = float(bin_val)

    return feat_dict

# Segmenta todas las partículas de una imagen y construye
# un DataFrame con sus características geométricas,
# cromáticas y texturales.
def build_particles_dataframe(bgr_img, is_sub_image=False):

    # Segmentación de partículas.
    contours, lab_img, bg_lab, filt_bgr, seg_meta = (
        segment_particles(
            bgr_img,
            is_sub_image=is_sub_image
        )
    )

    # Extracción de características para cada partícula.
    feature_rows = [
        {
            **extract_geometric_features(cnt),
            **extract_appearance_and_texture_features(
                cnt,
                filt_bgr,
                lab_img,
                bg_lab
            )
        }
        for cnt in contours
    ]

    # Conversión de las características a DataFrame.
    return pd.DataFrame(feature_rows), contours, seg_meta

print(
    "\n=== SECCIÓN 3: PROMEDIOS DE CARACTERÍSTICAS "
    "POR CLASE (IMÁGENES ORIGINALES) ==="
)

# Almacena los DataFrames de cada clase.
sample_dfs = []

# Procesa cada tipo de microplástico.
for cls_key in ["FRA", "PEL", "TAR"]:

    # Carga la imagen original.
    img_orig = read_and_scale_training_image(
        FULL_TRAIN_PATHS[cls_key],
        is_sub_image=False
    )

    # Obtiene las características de las partículas.
    df_orig, _, _ = build_particles_dataframe(
        img_orig,
        is_sub_image=False
    )

    # Etiqueta las muestras con su clase.
    df_orig["label"] = cls_key

    sample_dfs.append(df_orig)

# Combina todos los datos en un único DataFrame.
sample_all_df = pd.concat(
    sample_dfs,
    ignore_index=True
)

# Descriptores más representativos para el análisis.
summary_cols = [
    "circularity",
    "ellipse_ratio",
    "extent",
    "L_ratio",
    "chroma_rel"
]

# Muestra los promedios por clase.
print(
    sample_all_df
    .groupby("label")[summary_cols]
    .mean()
    .round(3)
    .to_string()
)

# Figura para visualizar la distribución de descriptores.
fig_desc, axes_desc = plt.subplots(
    1,
    3,
    figsize=(15, 4.5)
)

# Distribución del brillo relativo.
sns.boxplot(
    data=sample_all_df,
    x="label",
    y="L_ratio",
    palette="Set2",
    ax=axes_desc[0]
)

axes_desc[0].set_title("Brillo Relativo (L_ratio)")
axes_desc[0].set_xlabel("Clase")

# Distribución del croma relativo.
sns.boxplot(
    data=sample_all_df,
    x="label",
    y="chroma_rel",
    palette="Set2",
    ax=axes_desc[1]
)

axes_desc[1].set_title("Croma Relativo (chroma_rel)")
axes_desc[1].set_xlabel("Clase")

# Distribución de la circularidad.
sns.boxplot(
    data=sample_all_df,
    x="label",
    y="circularity",
    palette="Set2",
    ax=axes_desc[2]
)

axes_desc[2].set_title("Circularidad")
axes_desc[2].set_xlabel("Clase")

# Título general de la figura.
plt.suptitle(
    "Distribución de Descriptores Clave en las Imágenes Base de Entrenamiento",
    fontsize=13
)

# Ajusta automáticamente la distribución de los gráficos.
plt.tight_layout()

# Muestra la figura por pantalla.
plt.show()

Sección 4: Aumento de Datos Fotométrico y Reescalado

In [ ]:
# Parámetros que definen el rango de variación aplicado
# durante el aumento de datos fotométrico.
AUGMENTATION_BOUNDS = {

    # Variación del contraste de la imagen.
    "contrast": (0.8, 1.2),

    # Incremento o reducción del brillo global.
    "brightness": (-20, 20),

    # Modificación independiente de los canales de color.
    "color_gain": (0.9, 1.1),

    # Corrección gamma para alterar la luminosidad.
    "gamma": (0.8, 1.25),

    # Intensidad del desenfoque gaussiano.
    "blur": (0.01, 1.2),

    # Desviación estándar del ruido gaussiano añadido.
    "noise": (0, 4),

    # Factor de ampliación o reducción de escala.
    "scale_zoom": (0.80, 1.25),
}


# Genera valores aleatorios uniformes dentro de los límites
# definidos para cada parámetro de aumento de datos.
def draw_uniform_param(param_name, size=None):

    # Recupera los límites inferior y superior.
    low_val, high_val = AUGMENTATION_BOUNDS[param_name]

    # Muestrea valores aleatorios dentro del intervalo.
    return np.random.uniform(low_val, high_val, size)


# Aplica transformaciones fotométricas aleatorias para generar
# nuevas muestras de entrenamiento a partir de una imagen base.
def apply_photometric_augmentation(bgr_img):

    # Conversión a coma flotante para evitar pérdidas
    # de precisión durante las transformaciones.
    work_img = bgr_img.astype(np.float32)

    # Aplica un desenfoque gaussiano aleatorio.
    work_img = cv2.GaussianBlur(
        work_img,
        (0, 0),
        draw_uniform_param("blur")
    )

    # Añade ruido gaussiano para simular variaciones
    # de adquisición o calidad de imagen.
    work_img = work_img + np.random.normal(
        0,
        draw_uniform_param("noise"),
        work_img.shape
    )

    # Modifica de forma aleatoria el contraste y brillo.
    work_img = (
        work_img * draw_uniform_param("contrast")
        + draw_uniform_param("brightness")
    )

    # Ajusta cada canal de color de manera independiente.
    work_img = (
        work_img *
        draw_uniform_param("color_gain", 3)
    )

    # Aplica una corrección gamma aleatoria.
    work_img = 255.0 * (
        np.clip(work_img, 0, 255) / 255.0
    ) ** draw_uniform_param("gamma")

    # Conversión final al formato estándar de imagen.
    return work_img.astype(np.uint8)

print("\n=== SECCIÓN 4: EJEMPLOS GENERADOS POR EL AUMENTO DE DATOS ===")

# Figura de 3 filas (una por clase) y 4 columnas
# (imagen original + tres versiones aumentadas).
fig_aug, axes_aug = plt.subplots(3, 4, figsize=(14, 8))

# Genera ejemplos para cada clase de microplástico.
for r_idx, cls_key in enumerate(["FRA", "PEL", "TAR"]):

    # Carga la imagen de entrenamiento original.
    base_img = read_and_scale_training_image(
        FULL_TRAIN_PATHS[cls_key],
        is_sub_image=False
    )

    # Selecciona una región representativa para visualizar
    # más fácilmente el efecto de las transformaciones.
    crop_patch = base_img[220:500, 340:720]

    # Construye una lista con el parche original y tres
    # versiones aumentadas generadas aleatoriamente.
    patch_list = (
        [crop_patch] +
        [
            apply_photometric_augmentation(crop_patch)
            for _ in range(3)
        ]
    )

    # Representa cada una de las muestras generadas.
    for c_idx, p_img in enumerate(patch_list):

        # Conversión de BGR a RGB para visualización.
        axes_aug[r_idx, c_idx].imshow(
            cv2.cvtColor(p_img, cv2.COLOR_BGR2RGB)
        )

        # Identifica si se trata de la imagen original
        # o de una variante aumentada.
        axes_aug[r_idx, c_idx].set_title(
            f"{cls_key} Original"
            if c_idx == 0
            else f"{cls_key} Variante {c_idx}"
        )

        # Oculta los ejes para facilitar la observación.
        axes_aug[r_idx, c_idx].axis("off")

# Título general de la figura.
plt.suptitle(
    "Muestras Originales y Variantes Fotométricas por Clase",
    fontsize=13
)

# Ajusta automáticamente la distribución de las subfiguras.
plt.tight_layout()

# Muestra los ejemplos generados.
plt.show()

Sección 5: Clasificadores Autónomos en Cascada

In [ ]:
# Espacio de búsqueda de hiperparámetros utilizado para
# explorar distintas combinaciones de umbrales y seleccionar
# automáticamente la que maximiza el rendimiento sobre los
# datos de entrenamiento.
PARAMETER_SEARCH_SPACE = {

    # Umbral máximo de brillo relativo para identificar TAR.
    "max_brightness": np.round(np.arange(0.45, 0.91, 0.05), 3),

    # Umbral máximo de diferencia cromática para TAR.
    "max_color": np.round(np.arange(4.0, 15.0, 1.0), 3),

    # Circularidad mínima exigida para clasificar como PEL.
    "min_circularity": np.round(np.arange(0.74, 0.91, 0.01), 3),

    # Relación máxima entre ejes de la elipse para PEL.
    "max_ellipse_ratio": np.round(np.arange(1.16, 1.30, 0.02), 3),

    # Ocupación mínima del rectángulo delimitador para PEL.
    "min_extent": np.round(np.arange(0.64, 0.74, 0.02), 3),
}

# Genera valores aleatorios uniformes dentro del espacio de
# búsqueda definido para cada parámetro.
def draw_uniform_param(param_name, size=None):

    # Obtiene el rango permitido para el parámetro.
    low_val, high_val = AUGMENTATION_BOUNDS[param_name]

    # Genera uno o varios valores aleatorios.
    return np.random.uniform(low_val, high_val, size)

# Clasificación mediante reglas en cascada empleando
# exclusivamente los umbrales aprendidos durante el ajuste.
def predict_cascade_rules(df, thresholds):

    # Calcula una corrección automática para compensar
    # diferencias de iluminación entre dispositivos.
    device_l_shift = max(
        0.0,
        float(
            df["L_ratio"].median() -
            thresholds["ref_median_L"]
        )
    )

    # Ajusta el brillo relativo de las muestras.
    adjusted_l_ratio = df["L_ratio"] - device_l_shift

    # Regla para TAR:
    # partículas oscuras y poco diferenciadas en color
    # respecto al fondo.
    is_tar = (
        (adjusted_l_ratio < thresholds["max_brightness"]) &
        (df["chroma_rel"] < thresholds["max_color"])
    )

    # Regla para PEL:
    # partículas con elevada circularidad o geometría
    # aproximadamente elíptica y compacta.
    is_pel = (~is_tar) & (
        (df["circularity"] > thresholds["min_circularity"]) |
        (
            (df["ellipse_ratio"] < thresholds["max_ellipse_ratio"]) &
            (df["extent"] > thresholds["min_extent"])
        )
    )

    # Clasificación final en cascada.
    return np.select(
        [is_tar, is_pel],
        ["TAR", "PEL"],
        default="FRA"
    )

# Aprende automáticamente los umbrales óptimos evaluando
# todas las combinaciones posibles definidas en la rejilla
# de búsqueda.
def fit_cascade_rules_vectorized(train_df):

    # Etiquetas reales de entrenamiento.
    y_true = train_df["label"].values

    # Máscaras lógicas por clase.
    mask_fra = (y_true == "FRA")
    mask_pel = (y_true == "PEL")
    mask_tar = (y_true == "TAR")

    # Número de muestras de cada clase.
    n_fra, n_pel, n_tar = (
        np.sum(mask_fra),
        np.sum(mask_pel),
        np.sum(mask_tar)
    )

    # Extracción vectorizada de descriptores.
    l_arr = train_df["L_ratio"].values[:, None]
    c_arr = train_df["chroma_rel"].values[:, None]
    circ_arr = train_df["circularity"].values[:, None]
    ell_arr = train_df["ellipse_ratio"].values[:, None]
    ext_arr = train_df["extent"].values[:, None]

    # Generación de todas las combinaciones posibles
    # para la clasificación de TAR.
    tar_grid = [
        (b, c)
        for b in PARAMETER_SEARCH_SPACE["max_brightness"]
        for c in PARAMETER_SEARCH_SPACE["max_color"]
    ]

    # Generación de todas las combinaciones posibles
    # para la clasificación de PEL.
    pel_grid = [
        (r, e, x)
        for r in PARAMETER_SEARCH_SPACE["min_circularity"]
        for e in PARAMETER_SEARCH_SPACE["max_ellipse_ratio"]
        for x in PARAMETER_SEARCH_SPACE["min_extent"]
    ]

    # Evaluación simultánea de todas las reglas TAR.
    b_vec = np.array([p[0] for p in tar_grid])[None, :]
    c_vec = np.array([p[1] for p in tar_grid])[None, :]

    is_tar_mat = (
        (l_arr < b_vec) &
        (c_arr < c_vec)
    )

    # Evaluación simultánea de todas las reglas PEL.
    r_vec = np.array([p[0] for p in pel_grid])[None, :]
    e_vec = np.array([p[1] for p in pel_grid])[None, :]
    x_vec = np.array([p[2] for p in pel_grid])[None, :]

    is_pel_raw_mat = (
        (circ_arr > r_vec) |
        (
            (ell_arr < e_vec) &
            (ext_arr > x_vec)
        )
    )

    # Inicialización de la mejor solución encontrada.
    best_score = -1.0
    best_combo = None

    # Valor de referencia para compensación luminosa.
    ref_med_l = float(train_df["L_ratio"].median())

    # Búsqueda exhaustiva de la mejor combinación.
    for idx_t, (b_val, c_val) in enumerate(tar_grid):

        tar_col = is_tar_mat[:, idx_t:idx_t + 1]

        pel_mat = (~tar_col) & is_pel_raw_mat
        fra_mat = (~tar_col) & (~pel_mat)

        # Recall de cada clase.
        rec_tar = np.sum(tar_col[mask_tar, :], axis=0) / n_tar
        rec_pel = np.sum(pel_mat[mask_pel, :], axis=0) / n_pel
        rec_fra = np.sum(fra_mat[mask_fra, :], axis=0) / n_fra

        # Balanced Accuracy media.
        bal_scores = (
            rec_tar +
            rec_pel +
            rec_fra
        ) / 3.0

        max_idx_p = int(np.argmax(bal_scores))

        # Conserva la mejor configuración encontrada.
        if bal_scores[max_idx_p] > best_score:

            best_score = float(bal_scores[max_idx_p])

            r_val, e_val, x_val = pel_grid[max_idx_p]

            best_combo = {
                "max_brightness": float(b_val),
                "max_color": float(c_val),
                "min_circularity": float(r_val),
                "max_ellipse_ratio": float(e_val),
                "min_extent": float(x_val),
                "ref_median_L": ref_med_l,
            }

    return best_combo, best_score

# Clasificador multinivel inspirado en la arquitectura SMACC.
# Utiliza un primer nivel para identificar PEL y un segundo
# nivel para diferenciar FRA y TAR.
class SMACCCascadeClassifierML:

    def __init__(self):

        # Descriptores geométricos utilizados en el nivel 1.
        self.geom_cols = [
            "circularity",
            "compactness",
            "extent",
            "aspect_ratio",
            "ellipse_ratio",
            "radius_ratio"
        ]

        # Descriptores cromáticos.
        self.color_cols = [
            "L_ratio",
            "chroma_rel"
        ]

        # Histograma completo LBP.
        self.lbp_cols = [
            f"LBP_{i}"
            for i in range(256)
        ]

        # Reducción de dimensionalidad de LBP mediante PCA.
        self.pca_lbp = PCA(
            n_components=7,
            random_state=SEED_VALUE
        )

        # Nivel 1:
        # Random Forest para separar PEL del resto.
        self.clf_lvl1 = make_pipeline(
            StandardScaler(),
            RandomForestClassifier(
                n_estimators=200,
                max_depth=4,
                class_weight="balanced",
                random_state=SEED_VALUE
            )
        )

        # Nivel 2:
        # SVM RBF para distinguir FRA y TAR.
        self.clf_lvl2 = make_pipeline(
            StandardScaler(),
            SVC(
                C=1.0,
                kernel="rbf",
                class_weight="balanced",
                random_state=SEED_VALUE
            )
        )

    def fit(self, df_train):

        # Extrae los descriptores geométricos que utilizará
        # el primer nivel de la cascada.
        x_geom = df_train[self.geom_cols].values

        # Convierte el problema original de tres clases en una
        # clasificación binaria: Pellet frente al resto.
        y_lvl1 = np.where(
            df_train["label"] == "PEL",
            "PEL",
            "REST"
        )

        # Entrena el clasificador de Nivel 1.
        self.clf_lvl1.fit(x_geom, y_lvl1)

        # Conserva únicamente las muestras que no son Pellet
        # para entrenar el segundo nivel de la cascada.
        df_rest = df_train[
            df_train["label"] != "PEL"
        ].copy()

        # Reduce la dimensionalidad de los 256 descriptores
        # LBP mediante PCA.
        x_lbp_pca = self.pca_lbp.fit_transform(
            df_rest[self.lbp_cols].values
        )

        # Construye el vector de características del Nivel 2
        # combinando geometría, color y textura reducida.
        x_lvl2 = np.hstack([
            df_rest[self.geom_cols].values,
            df_rest[self.color_cols].values,
            x_lbp_pca
        ])

        # Entrena el clasificador encargado de distinguir
        # entre Fragmentos (FRA) y Alquitrán (TAR).
        self.clf_lvl2.fit(
            x_lvl2,
            df_rest["label"].values
        )

        return self


    def predict(self, df_eval):

        # Predicción inicial del Nivel 1 para determinar
        # si una partícula corresponde a Pellet.
        pred_lvl1 = self.clf_lvl1.predict(
            df_eval[self.geom_cols].values
        )

        # Proyección de los descriptores LBP sobre el espacio
        # PCA aprendido durante el entrenamiento.
        x_lbp_pca = self.pca_lbp.transform(
            df_eval[self.lbp_cols].values
        )

        # Construcción del vector de características empleado
        # por el clasificador de Nivel 2.
        x_lvl2 = np.hstack([
            df_eval[self.geom_cols].values,
            df_eval[self.color_cols].values,
            x_lbp_pca
        ])

        # Predicción del segundo nivel para diferenciar
        # entre Fragmentos y Alquitrán.
        pred_lvl2 = self.clf_lvl2.predict(x_lvl2)

        # Decisión final de la cascada:
        # 1. Si el Nivel 2 predice TAR, se asigna TAR.
        # 2. En caso contrario, si el Nivel 1 predice PEL,
        #    se asigna PEL.
        # 3. El resto de muestras se clasifican como FRA.
        return np.where(
            pred_lvl2 == "TAR",
            "TAR",
            np.where(
                pred_lvl1 == "PEL",
                "PEL",
                "FRA"
            )
        )
    

Sección 6: Entrenamiento del Modelo y Visualización de Umbrales Aprendidos

In [ ]:
# Reinicia la semilla aleatoria para garantizar que las
# transformaciones aplicadas sean reproducibles.
np.random.seed(SEED_VALUE)

# Lista donde se almacenarán las características extraídas
# de todas las partículas de entrenamiento.
training_Part_list = []

# Procesa las imágenes completas de cada clase.
for cls_key, file_path in FULL_TRAIN_PATHS.items():

    # Carga la imagen original.
    base_img = read_and_scale_training_image(
        file_path,
        is_sub_image=False
    )

    # Genera variantes fotométricas de la imagen original
    # para aumentar la diversidad del conjunto de entrenamiento.
    augmented_versions = (
        [base_img] +
        [
            apply_photometric_augmentation(base_img)
            for _ in range(AUGMENTATION_COUNT)
        ]
    )

    # Extrae características de cada versión generada.
    for ver_img in augmented_versions:

        df_part, _, _ = build_particles_dataframe(
            ver_img,
            is_sub_image=False
        )

        # Simula pequeñas variaciones de escala modificando
        # únicamente los descriptores geométricos dependientes
        # del tamaño físico.
        zoom_factor = float(
            draw_uniform_param("scale_zoom")
        )

        # El área varía con el cuadrado de la escala.
        df_part["area"] = (
            df_part["area"] *
            (zoom_factor ** 2)
        )

        # El perímetro varía linealmente con la escala.
        df_part["perimeter"] = (
            df_part["perimeter"] *
            zoom_factor
        )

        # Asigna la etiqueta correspondiente a la clase.
        df_part["label"] = cls_key

        training_Part_list.append(df_part)

# Procesa las subimágenes de referencia de cada clase.
for cls_key, file_path in SUB_TRAIN_PATHS.items():

    # Carga la subimagen original.
    sub_img = read_and_scale_training_image(
        file_path,
        is_sub_image=True
    )

    # Utiliza la imagen original y una versión aumentada.
    for ver_img in [
        sub_img,
        apply_photometric_augmentation(sub_img)
    ]:

        df_part, _, _ = build_particles_dataframe(
            ver_img,
            is_sub_image=True
        )

        # Etiqueta los ejemplos con la clase correspondiente.
        df_part["label"] = cls_key

        training_Part_list.append(df_part)

# Construye el DataFrame final unificando todas las partículas.
train_df = pd.concat(
    training_Part_list,
    ignore_index=True
)

# Aprende automáticamente los umbrales óptimos utilizados
# por la cascada basada en reglas.
learned_thresholds, train_bal_score = (
    fit_cascade_rules_vectorized(train_df)
)

# Entrena el clasificador multinivel basado en
# Machine Learning.
smacc_ml_model = (
    SMACCCascadeClassifierML()
    .fit(train_df)
)

print("\n=== SECCIÓN 6: RESULTADO DEL ENTRENAMIENTO AUTÓNOMO ===")

# Muestra el número de partículas disponibles por clase.
print(
    "Partículas de entrenamiento por clase:",
    train_df["label"].value_counts().to_dict()
)

# Presenta los umbrales aprendidos automáticamente.
print(
    "Umbrales aprendidos autónomamente "
    "(sin constantes fijas):"
)

for param_k, param_v in learned_thresholds.items():
    print(f"  - {param_k}: {param_v:.3f}")

# Muestra el Balanced Accuracy obtenido durante
# la fase de entrenamiento.
print(
    f"Acierto equilibrado en entrenamiento: "
    f"{train_bal_score:.3f}"
)

# Figura con dos gráficos para representar visualmente
# las reglas aprendidas durante el entrenamiento.
fig_tr, axes_tr = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

# Colores asociados a cada clase.
color_palette = {
    "FRA": "#e74c3c",
    "PEL": "#2ecc71",
    "TAR": "#3498db"
}

# Representación de las muestras utilizando los dos
# descriptores empleados para distinguir partículas TAR.
sns.scatterplot(
    data=train_df,
    x="L_ratio",
    y="chroma_rel",
    hue="label",
    palette=color_palette,
    alpha=0.5,
    ax=axes_tr[0]
)

# Umbral máximo aprendido para brillo relativo.
axes_tr[0].axvline(
    learned_thresholds["max_brightness"],
    color="black",
    linestyle="--",
    label="max_brightness"
)

# Umbral máximo aprendido para diferencia cromática.
axes_tr[0].axhline(
    learned_thresholds["max_color"],
    color="navy",
    linestyle=":",
    label="max_color"
)

# Configuración de etiquetas y título.
axes_tr[0].set_title(
    "Frontera Aprendida para Alquitrán (TAR)"
)

axes_tr[0].set_xlabel(
    "Brillo Relativo (L_ratio)"
)

axes_tr[0].set_ylabel(
    "Croma Relativo (chroma_rel)"
)

axes_tr[0].legend()

# Representación de las muestras no clasificadas como TAR
# utilizando los descriptores geométricos más relevantes.
sns.scatterplot(
    data=train_df[train_df["label"] != "TAR"],
    x="ellipse_ratio",
    y="circularity",
    hue="label",
    palette={
        "FRA": "#e74c3c",
        "PEL": "#2ecc71"
    },
    alpha=0.5,
    ax=axes_tr[1]
)

# Umbral mínimo aprendido para circularidad.
axes_tr[1].axhline(
    learned_thresholds["min_circularity"],
    color="darkgreen",
    linestyle="--",
    label="min_circularity"
)

# Umbral máximo aprendido para relación elíptica.
axes_tr[1].axvline(
    learned_thresholds["max_ellipse_ratio"],
    color="purple",
    linestyle=":",
    label="max_ellipse_ratio"
)

# Configuración de etiquetas y título.
axes_tr[1].set_title(
    "Frontera Aprendida para Pellets (PEL) vs Fragmentos (FRA)"
)

axes_tr[1].set_xlabel(
    "Relación de Ejes de Elipse (ellipse_ratio)"
)

axes_tr[1].set_ylabel(
    "Circularidad"
)

axes_tr[1].legend()

# Título general de la figura.
plt.suptitle(
    "Fronteras de Decisión Aprendidas Autónomamente "
    "sobre los Datos de Entrenamiento",
    fontsize=13
)

# Ajusta automáticamente la distribución de los gráficos.
plt.tight_layout()

# Muestra los resultados obtenidos.
plt.show()

Sección 7: Evaluación en la Imagen de Test (MPs_test.jpg)

In [ ]:
# Carga las anotaciones de referencia almacenadas
# en el archivo CSV de evaluación.
def read_annotations_csv(csv_path):

    # Construye la ruta completa del archivo CSV.
    csv_full_path = os.path.join("images", csv_path)

    # Comprueba que el archivo existe.
    if not os.path.exists(csv_full_path):
        raise FileNotFoundError(
            f"No se encontró el archivo CSV: {csv_full_path}"
        )

    # Lee todas las anotaciones y las devuelve
    # como una lista de diccionarios.
    with open(
        csv_full_path,
        mode="r",
        newline="",
        encoding="utf-8"
    ) as f_in:
        return list(csv.DictReader(f_in))

# Calcula la matriz completa de solapamientos IoU
# entre las cajas reales y las cajas detectadas.
def compute_pairwise_iou(gt_boxes, det_boxes):

    # Coordenadas de intersección entre cada pareja
    # de cajas reales y detectadas.
    inter_x0 = np.maximum(
        gt_boxes[:, None, 0],
        det_boxes[None, :, 0]
    )

    inter_y0 = np.maximum(
        gt_boxes[:, None, 1],
        det_boxes[None, :, 1]
    )

    inter_x1 = np.minimum(
        gt_boxes[:, None, 2],
        det_boxes[None, :, 2]
    )

    inter_y1 = np.minimum(
        gt_boxes[:, None, 3],
        det_boxes[None, :, 3]
    )

    # Anchura y altura de cada intersección.
    inter_w = np.maximum(0.0, inter_x1 - inter_x0)
    inter_h = np.maximum(0.0, inter_y1 - inter_y0)

    # Área de intersección.
    inter_area = inter_w * inter_h

    # Área de las cajas reales.
    gt_area = (
        (gt_boxes[:, 2] - gt_boxes[:, 0]) *
        (gt_boxes[:, 3] - gt_boxes[:, 1])
    )

    # Área de las cajas detectadas.
    det_area = (
        (det_boxes[:, 2] - det_boxes[:, 0]) *
        (det_boxes[:, 3] - det_boxes[:, 1])
    )

    # Área de unión aplicada en la fórmula IoU.
    union_area = (
        gt_area[:, None] +
        det_area[None, :] -
        inter_area
    )

    # Devuelve el IoU para cada pareja de cajas.
    return np.where(
        union_area > 0,
        inter_area / union_area,
        0.0
    )

# Asocia cada anotación real con la detección que
# presenta el mayor solapamiento IoU.
def match_detections_to_annotations(
    det_df,
    annotations_list,
    min_iou=MIN_IOU_THRESHOLD
):

    # Conversión de las anotaciones reales a formato
    # numérico para facilitar el cálculo del IoU.
    gt_boxes = np.array([
        [
            int(ann["x_min"]),
            int(ann["y_min"]),
            int(ann["x_max"]),
            int(ann["y_max"])
        ]
        for ann in annotations_list
    ], dtype=np.float64)

    # Conversión de los contornos detectados a cajas.
    det_boxes = np.column_stack([
        det_df["x"].values,
        det_df["y"].values,
        det_df["x"].values + det_df["w"].values,
        det_df["y"].values + det_df["h"].values
    ]).astype(np.float64)

    # Matriz de solapamientos.
    iou_mat = compute_pairwise_iou(
        gt_boxes,
        det_boxes
    )

    # Diccionario de emparejamientos válidos.
    matched_dict = {}

    # Busca para cada anotación la mejor detección.
    for gt_idx, ann in enumerate(annotations_list):

        # Evita reutilizar una detección ya asignada.
        available_indices = [
            i
            for i in range(len(det_df))
            if i not in matched_dict
        ]

        if not available_indices:
            break

        # Selecciona la detección con mayor IoU.
        best_det = max(
            available_indices,
            key=lambda i: iou_mat[gt_idx, i]
        )

        # Acepta el emparejamiento si supera el umbral.
        if iou_mat[gt_idx, best_det] >= min_iou:
            matched_dict[best_det] = ann["label"]

    # Construye el DataFrame de detecciones emparejadas.
    matched_df = det_df.loc[
        list(matched_dict.keys())
    ].copy()

    matched_df["label"] = list(
        matched_dict.values()
    )

    return matched_df

# Genera un resumen numérico del rendimiento
# de detección y clasificación.
def summarize_detection_performance(
    det_df,
    matched_df,
    annotations_list
):

    return {

        # Número total de objetos anotados.
        "anotadas": len(annotations_list),

        # Número total de objetos detectados.
        "detectadas": len(det_df),

        # Detecciones correctamente emparejadas.
        "emparejadas": len(matched_df),

        # Objetos reales que no fueron detectados.
        "no detectadas":
            len(annotations_list) - len(matched_df),

        # Detecciones sin correspondencia real.
        "falsos positivos":
            len(det_df) - len(matched_df),

        # Clasificaciones correctas.
        "correctas":
            int(
                (matched_df["label"] ==
                 matched_df["pred"]).sum()
            ),
    }

# Devuelve el color utilizado para representar cada
# detección durante la visualización de resultados.
def select_contour_bgr_color(det_idx, matched_df):

    # Detección sin correspondencia real.
    if det_idx not in matched_df.index:
        return (255, 0, 0)

    # Clasificación correcta.
    if (
        matched_df.at[det_idx, "label"] ==
        matched_df.at[det_idx, "pred"]
    ):
        return (0, 255, 0)

    # Clasificación incorrecta.
    return (0, 0, 255)

# Calcula métricas globales de clasificación y
# representa gráficamente la matriz de confusión.
def display_evaluation_metrics(
    y_true,
    y_pred,
    plot_title="Matriz de Confusión"
):

    # Etiquetas utilizadas en la evaluación.
    class_labels = ["FRA", "PEL", "TAR"]

    # Métricas globales de clasificación.
    print(
        f"Accuracy          = "
        f"{accuracy_score(y_true, y_pred):.3f}"
    )

    print(
        f"Balanced Accuracy = "
        f"{balanced_accuracy_score(y_true, y_pred):.3f}"
    )

    print(
        f"Precision         = "
        f"{precision_score(y_true, y_pred, average='weighted', zero_division=0):.3f}"
    )

    print(
        f"Recall            = "
        f"{recall_score(y_true, y_pred, average='weighted', zero_division=0):.3f}"
    )

    print(
        f"F1 Score          = "
        f"{f1_score(y_true, y_pred, average='weighted', zero_division=0):.3f}\n"
    )

    # Informe detallado por clase.
    print(
        classification_report(
            y_true,
            y_pred,
            labels=class_labels,
            zero_division=0
        )
    )

    # Construcción de la matriz de confusión.
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=class_labels
    )

    # Representación gráfica de la matriz.
    plt.figure(figsize=(6, 6))

    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cbar=False,
        cmap="Blues",
        xticklabels=class_labels,
        yticklabels=class_labels,
        annot_kws={"size": 16}
    )

    ax.set_xlabel("Predicha")
    ax.set_ylabel("Real/Anotado")
    ax.set_title(plot_title)

    plt.tight_layout()
    plt.show()

    return cm

# Carga la imagen de evaluación.
test_bgr = read_image_file(TEST_IMG_FILE)

# Segmenta las partículas y extrae sus características.
test_df, test_contours, test_seg_meta = build_particles_dataframe(
    test_bgr,
    is_sub_image=False
)

# Carga las anotaciones reales utilizadas como referencia.
annotations_gt = read_annotations_csv(TEST_CSV_FILE)

# Empareja cada detección con la anotación que presenta
# mayor solapamiento espacial (IoU).
matched_test_df = match_detections_to_annotations(
    test_df,
    annotations_gt,
    min_iou=MIN_IOU_THRESHOLD
)

# Obtiene la clase predicha por la cascada heurística.
matched_test_df["pred"] = predict_cascade_rules(
    matched_test_df,
    learned_thresholds
)

# Etiquetas reales y predichas para calcular métricas.
y_true_test = matched_test_df["label"].values
y_pred_test = matched_test_df["pred"].values

print("\n=== SECCIÓN 7: RESUMEN DE EVALUACIÓN EN TEST (MPs_test.jpg) ===")

# Resumen global de detección y clasificación.
test_summary = summarize_detection_performance(
    test_df,
    matched_test_df,
    annotations_gt
)

print(test_summary)

print(
    f"Falsos negativos (objetos anotados no detectados): "
    f"{test_summary['no detectadas']}"
)

# Muestra las métricas de clasificación y la matriz
# de confusión correspondiente.
display_evaluation_metrics(
    y_true_test,
    y_pred_test,
    plot_title="Matriz de Confusión - CIE Lab + Sauvola + Cascada Autónoma"
)

# Colores utilizados para representar cada clase anotada.
gt_colors_bgr = {
    "FRA": (0, 0, 255),
    "PEL": (0, 200, 0),
    "TAR": (255, 0, 0)
}

# Copia de la imagen para dibujar las anotaciones reales.
vis_gt_boxes = test_bgr.copy()

# Dibuja las cajas delimitadoras y etiquetas definidas en el CSV.
for ann in annotations_gt:

    # Selecciona el color correspondiente a la clase.
    box_col = gt_colors_bgr.get(
        ann["label"],
        (0, 0, 0)
    )

    x_min = int(ann["x_min"])
    y_min = int(ann["y_min"])
    x_max = int(ann["x_max"])
    y_max = int(ann["y_max"])

    # Representa la anotación sobre la imagen, es decir, dibuja la caja delimitadora.
    cv2.rectangle(
        vis_gt_boxes,
        (x_min, y_min),
        (x_max, y_max),
        box_col,
        2
    )

    # Muestra la etiqueta asociada.
    cv2.putText(
        vis_gt_boxes,
        ann["label"],
        (x_min, max(15, y_min - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        box_col,
        1,
        cv2.LINE_AA
    )

# Copia de la imagen para mostrar las clases predichas.
vis_pred_boxes = test_bgr.copy()

# Colores utilizados para representar cada clase predicha.
pred_colors_bgr = {
    "FRA": (0, 0, 255),
    "PEL": (0, 200, 0),
    "TAR": (255, 0, 0)
}

# Dibuja las cajas y etiquetas predichas.
for _, row in matched_test_df.iterrows():

    pred_color = pred_colors_bgr.get(
        row["pred"],
        (0, 0, 0)
    )

    x_min = int(row["x"])
    y_min = int(row["y"])
    x_max = int(row["x"] + row["w"])
    y_max = int(row["y"] + row["h"])

    cv2.rectangle(
        vis_pred_boxes,
        (x_min, y_min),
        (x_max, y_max),
        pred_color,
        2
    )

    cv2.putText(
        vis_pred_boxes,
        row["pred"],
        (x_min, max(15, y_min - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        pred_color,
        1,
        cv2.LINE_AA
    )

# Copia de la imagen para mostrar el resultado
# de la evaluación de las detecciones.
vis_eval_contours = test_bgr.copy()

# Dibuja cada contorno detectado utilizando un color
# que indica si la clasificación fue correcta o no.
for idx_c, cnt in enumerate(test_contours):

    cv2.drawContours(
        vis_eval_contours,
        [cnt],
        -1,
        select_contour_bgr_color(
            idx_c,
            matched_test_df
        ),
        2
    )

# Los cuatro paneles permiten inspeccionar visualmente
# todo el flujo de detección y clasificación.
fig_t, axes_t = plt.subplots(
    1,
    4,
    figsize=(20, 8)
)

# Visualización del mapa de pseudo-transparencia
# obtenido tras la estimación del fondo en CIE Lab.
axes_t[0].imshow(
    test_seg_meta["pseudo_trans"],
    cmap="gray"
)

axes_t[0].set_title(
    "1. Pseudo-Transparencia CIE Lab\n(Fondo sin halos ni bandas)"
)

axes_t[0].axis("off")

# Visualización de la máscara binaria final resultante
# de la combinación Sauvola + Otsu local.
axes_t[1].imshow(
    test_seg_meta["mask_clean"],
    cmap="gray"
)

axes_t[1].set_title(
    f"2. Contornos Sauvola + Otsu Local\n"
    f"({len(test_contours)} detectados | "
    f"m={test_seg_meta['sauvola_mean']:.1f})"
)

axes_t[1].axis("off")

# Visualización de las anotaciones reales del conjunto
# de evaluación.
axes_t[2].imshow(
    cv2.cvtColor(
        vis_gt_boxes,
        cv2.COLOR_BGR2RGB
    )
)

axes_t[2].set_title(
    f"3. Anotaciones CSV ({len(annotations_gt)})\n"
    "Rojo=FRA | Verde=PEL | Azul=TAR"
)

axes_t[2].axis("off")

# Visualización de las predicciones del clasificador
axes_t[3].imshow(
    cv2.cvtColor(
        vis_pred_boxes,
        cv2.COLOR_BGR2RGB
    )
)

axes_t[3].set_title(
    "4. Clases Predichas\n"
    "Rojo=FRA | Verde=PEL | Azul=TAR"
)

axes_t[3].axis("off")

# Ajusta la distribución de los paneles para evitar
# solapamientos entre títulos y figuras.
plt.tight_layout()

# Muestra la visualización final.
plt.show()

# Crea una figura independiente para visualizar de forma
# clara los aciertos, errores de clasificación y falsos positivos.
fig_eval, ax_eval = plt.subplots(
    figsize=(10, 8)
)

# Muestra la imagen de evaluación con los contornos
# coloreados según el resultado de la clasificación.
ax_eval.imshow(
    cv2.cvtColor(
        vis_eval_contours,
        cv2.COLOR_BGR2RGB
    )
)

# Resume visualmente el rendimiento obtenido en test:
# - Aciertos: partículas clasificadas correctamente.
# - FP: detecciones sin correspondencia real.
# - FN: partículas anotadas que no fueron detectadas.
ax_eval.set_title(
    f"Evaluación de Clasificación\n"
    f"Aciertos={test_summary['correctas']} | "
    f"FP={test_summary['falsos positivos']} | "
    f"FN={test_summary['no detectadas']}\n"
    "Verde=Acierto | Rojo=Fallo | Azul=FP"
)

# Oculta los ejes para centrar la atención en la evaluación.
ax_eval.axis("off")

# Ajusta automáticamente los márgenes de la figura.
plt.tight_layout()

# Muestra la visualización final de evaluación.
plt.show()


Sección 8: Comparativa con Otros Clasificadores y Gráfica de Resultados

In [ ]:
# Todos los clasificadores convencionales utilizan exactamente
# el mismo subconjunto de características para garantizar una
# comparación justa entre modelos.

# Conjunto reducido de características utilizado por los
# clasificadores supervisados convencionales.
COMPARISON_FEATURES = [
    "circularity",
    "compactness",
    "extent",
    "ellipse_ratio",
    "L_ratio",
    "chroma_rel"
]

# Modelos utilizados para comparar el rendimiento de la
# cascada propuesta frente a alternativas clásicas.
BENCHMARK_CLASSIFIERS = {

    # Arquitectura multinivel desarrollada en este trabajo.
    "Cascada SMACC (RF Geom + SVM Color/LBP)": smacc_ml_model,

    # Clasificador basado en vecinos más próximos.
    "KNN (K=3)": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=3)
    ),

    # Árbol de decisión de profundidad limitada.
    "Árbol de Decisión (prof. 3)": DecisionTreeClassifier(
        max_depth=3,
        class_weight="balanced",
        random_state=SEED_VALUE
    ),

    # Máquina de vectores soporte con núcleo RBF.
    "SVM (RBF)": make_pipeline(
        StandardScaler(),
        SVC(
            class_weight="balanced",
            random_state=SEED_VALUE
        )
    ),

    # Bosque aleatorio basado en múltiples árboles.
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=SEED_VALUE
    ),
}

# Resultados iniciales correspondientes a la cascada
# heurística basada en reglas aprendidas automáticamente.
comparison_rows = [{
    "modelo": "Reglas en cascada autónomas (Lab + Sauvola)",

    # Balanced Accuracy obtenida durante el aprendizaje
    # de los umbrales heurísticos.
    "balanced (train)": train_bal_score,

    # Accuracy obtenida sobre la imagen de test.
    "accuracy": accuracy_score(
        y_true_test,
        y_pred_test
    ),

    # Balanced Accuracy obtenida sobre la imagen de test.
    "balanced": balanced_accuracy_score(
        y_true_test,
        y_pred_test
    ),

    # Medida F1 ponderada.
    "f1": f1_score(
        y_true_test,
        y_pred_test,
        average="weighted"
    ),

    # Número total de clasificaciones correctas.
    "correctas": int(
        np.sum(y_true_test == y_pred_test)
    ),
}]

# Entrena y evalúa cada clasificador de referencia.
for model_name, clf_obj in BENCHMARK_CLASSIFIERS.items():

    # La cascada SMACC ya fue entrenada previamente.
    if "Cascada SMACC" in model_name:

        pred_tr = clf_obj.predict(train_df)
        pred_te = clf_obj.predict(matched_test_df)

    else:

        # Entrenamiento del clasificador utilizando
        # el subconjunto común de características.
        clf_obj.fit(
            train_df[COMPARISON_FEATURES],
            train_df["label"]
        )

        # Predicciones sobre entrenamiento.
        pred_tr = clf_obj.predict(
            train_df[COMPARISON_FEATURES]
        )

        # Predicciones sobre test.
        pred_te = clf_obj.predict(
            matched_test_df[COMPARISON_FEATURES]
        )

    # Almacena las métricas obtenidas por el modelo.
    comparison_rows.append({
        "modelo": model_name,

        "balanced (train)":
            balanced_accuracy_score(
                train_df["label"],
                pred_tr
            ),

        "accuracy":
            accuracy_score(
                y_true_test,
                pred_te
            ),

        "balanced":
            balanced_accuracy_score(
                y_true_test,
                pred_te
            ),

        "f1":
            f1_score(
                y_true_test,
                pred_te,
                average="weighted"
            ),

        # Número total de partículas clasificadas correctamente.
        "correctas":
            int(
                np.sum(y_true_test == pred_te)
            ),
    })

# Construye la tabla comparativa final.
results_table = pd.DataFrame(comparison_rows)

print("\n=== SECCIÓN 8: TABLA COMPARATIVA DE CLASIFICADORES ===")

# Muestra la tabla resumen con todas las métricas.
print(
    results_table
    .round(3)
    .to_string(index=False)
)

# Etiquetas abreviadas para mejorar la legibilidad
# de la gráfica comparativa.
plot_labels = [
    "Reglas",
    "SMACC",
    "KNN",
    "Árbol",
    "SVM",
    "RF"
]

# Figura con dos gráficas resumen del rendimiento.
fig_cmp, axes_cmp = plt.subplots(
    1,
    2,
    figsize=(17, 6)
)

# Posiciones utilizadas en el eje X.
x_indices = np.arange(len(results_table))

# Anchura de las barras.
width_b = 0.35

# Comparación entre el rendimiento en entrenamiento
# y el rendimiento sobre la imagen de test.
axes_cmp[0].bar(
    x_indices - width_b / 2,
    results_table["balanced (train)"],
    width=width_b,
    label="Balanced (Entrenamiento)",
    color="#5dade2"
)

axes_cmp[0].bar(
    x_indices + width_b / 2,
    results_table["balanced"],
    width=width_b,
    label="Balanced (Test)",
    color="#2ecc71"
)

axes_cmp[0].set_xticks(x_indices)

axes_cmp[0].set_xticklabels(
    plot_labels
)

axes_cmp[0].set_ylim(0, 1.08)

axes_cmp[0].set_ylabel(
    "Acierto Equilibrado (Balanced Accuracy)"
)

axes_cmp[0].set_title(
    "Rendimiento en Entrenamiento vs Test"
)

axes_cmp[0].legend(
    loc="lower left"
)

# Comparación del número total de aciertos y del número
# de partículas TAR clasificadas correctamente.
bars_total = axes_cmp[1].barh(
    results_table["modelo"],
    results_table["correctas"],
    color="#3498db",
    label=f"Clasificaciones Correctas (/{len(y_true_test)})"
)

axes_cmp[1].set_xlim(0, len(y_true_test) + 10)

axes_cmp[1].set_xlabel(
    "Número de Clasificaciones Correctas"
)

axes_cmp[1].set_title(
    "Clasificaciones Correctas por Modelo"
)

# Sitúa el mejor resultado en la parte superior.
axes_cmp[1].invert_yaxis()

axes_cmp[1].legend(
    loc="lower left"
)

# Añade etiquetas numéricas junto a cada barra
# para facilitar la interpretación visual.
for bar_t in bars_total:

    val_w = int(bar_t.get_width())

    axes_cmp[1].text(
        val_w + 1.2,
        bar_t.get_y() + bar_t.get_height() / 2,
        f"{val_w}",
        va="center",
        fontweight="bold"
    )

# Título general de la comparativa.
plt.suptitle(
    "Comparativa Global de Clasificadores en MPs_test.jpg",
    fontsize=13
)

# Ajusta automáticamente la distribución de la figura.
plt.tight_layout()

# Muestra los resultados gráficos.
plt.show()